In [2]:
# Block 1: Imports and Constants
# ===============================
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate
# --- NEW: Import Focal Loss (Make sure you install it: pip install focal-loss-pytorch) ---
# If using a different implementation, adjust the import and instantiation
try:
    from focal_loss.focal_loss import FocalLoss
    print("Using FocalLoss from focal-loss-pytorch")
    # For multi-label, need sigmoid reduction. Alpha can balance pos/neg. Gamma focuses on hard examples.
    # Adjust alpha (e.g., 0.25) and gamma (e.g., 2.0) based on experimentation
    focal_loss_criterion = FocalLoss(gamma=2.0, alpha=0.25, reduction='mean')
except ImportError:
    print("WARNING: focal-loss-pytorch not found. Falling back to BCEWithLogitsLoss.")
    print("Install it via: pip install focal-loss-pytorch")
    print("For training, object loss will use BCEWithLogitsLoss unless you install FocalLoss.")
    focal_loss_criterion = nn.BCEWithLogitsLoss() # Fallback

# --- Constants (mostly unchanged) ---
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_objects_reduced.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16

# --- Tokenizer and PAD_ID (unchanged) ---
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size
NUM_COLORS = 77
NUM_CATEGORIES = 53
NUM_OBJECTS = 61

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Install it via: pip install focal-loss-pytorch
For training, object loss will use BCEWithLogitsLoss unless you install FocalLoss.
Using device: cuda


In [3]:
# Block 2: Granger Causality (Unchanged)
# =======================================
def create_granger_causality_matrix(eeg_batch):
    # --- (Keep your existing Granger function code here) ---
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            # Ensure minimum length for the test
            min_len = 20 # Adjust if needed, depends on maxlag
            if len(ts_i) < min_len or len(ts_j) < min_len:
                 causality_matrix[i, j] = 0.0
                 continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                # Use a smaller maxlag if sequences are short, ensure it's less than len(data)/2 - 1
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                # Get p-value for the chosen lag (e.g., the maxlag tested)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e: # Catch potential errors during test
                # print(f"Granger test failed for channels {i}-{j}: {e}") # Optional: for debugging
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    # Ensure edge_attr has the correct shape, even if empty
    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0: # Handle scalar case if only one edge exists
         edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [4]:
# Block 3: Dataset and DataLoader (Unchanged)
# ===========================================
class EEGMetaTextH5Dataset(Dataset):
    # --- (Keep your existing Dataset code here) ---
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            # Open in read-only mode
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32)) # Keep as float for FocalLoss/BCE
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    # --- (Keep your existing collate_fn code here) ---
    eeg_list, meta_list, text_list = [], [], []
    max_text_len = 0
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)
        if len(txt) > max_text_len:
            max_text_len = len(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)

    # Pad text sequences manually or use pad_sequence
    # Ensure text tensors have the same length before stacking if needed elsewhere,
    # but pad_sequence handles variable lengths correctly.
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

# --- Create Dataset and Loaders (Unchanged) ---
dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
N = len(dataset)
n_train = int(N * TRAIN_PCT)
n_val   = int(N * VAL_PCT)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

# --- Execute Granger Matrix Creation (Unchanged, but ensure it runs) ---
print("Attempting to create Granger matrix...")
try:
    eeg_b, _, _ = next(iter(train_loader))
    print(f"Creating a static Granger Causality matrix on {device}...")
    granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
    num_channels = eeg_b.shape[1] # Should be 62

    # Add self-loops to handle empty graphs or ensure connectivity
    print("Original edge_index shape:", granger_edge_index.shape)
    print("Original edge_attr shape:", granger_edge_attr.shape)

    # Ensure edge_attr is correctly shaped before add_self_loops
    if granger_edge_attr is not None and granger_edge_attr.ndim == 0:
        granger_edge_attr = granger_edge_attr.unsqueeze(0) # Make it 1D if scalar

    granger_edge_index, granger_edge_attr = add_self_loops(
        granger_edge_index,
        edge_attr=granger_edge_attr,
        num_nodes=num_channels,
        fill_value=1.0 # Or use None if edge_attr should be None after self-loops
    )

    # Handle case where edge_attr becomes None after add_self_loops
    if granger_edge_attr is None:
         print("Warning: edge_attr became None after add_self_loops. Creating ones.")
         granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)


    # Correct the data types before moving to the device
    granger_edge_index = granger_edge_index.to(torch.long)
    granger_edge_attr = granger_edge_attr.to(torch.float32) # Ensure float32

    granger_edge_index = granger_edge_index.to(device)
    granger_edge_attr = granger_edge_attr.to(device)

    print("Granger Matrix created/processed and loaded to device.")
    print("Final edge_index shape:", granger_edge_index.shape)
    print("Final edge_attr shape:", granger_edge_attr.shape)

except Exception as e:
    print(f"Error during Granger matrix creation: {e}")
    print("Falling back to a fully connected graph with self-loops.")
    # Fallback: create a fully connected graph + self-loops if Granger fails
    num_channels = 62 # Assuming 62 channels if loader fails
    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1) # Make bidirectional
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)

    granger_edge_index = edge_index.to(torch.long).to(device)
    granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device) # Ensure float32
    print("Fallback graph created.")

Attempting to create Granger matrix...
Creating a static Granger Causality matrix on cuda...


/home/poorna/venvs/torch/lib64/python3.11/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


Original edge_index shape: torch.Size([2, 1923])
Original edge_attr shape: torch.Size([1923])
Granger Matrix created/processed and loaded to device.
Final edge_index shape: torch.Size([2, 1985])
Final edge_attr shape: torch.Size([1985])


In [5]:
# Block 4: Model Components (Encoder, Attention, MetaEncoder - Unchanged)
# =========================================================================
class SpatioTemporalEEGEncoder(nn.Module):
    # --- (Keep your existing Encoder code here) ---
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        # Corrected GRU input dimension logic - GCN output has enc_hidden features
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True) # Use batch_first=True for easier handling
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        # eeg shape: [batch_size, num_channels, num_timesteps]
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        # Prepare graph data for batch processing
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size) # Repeat edge attributes for each graph in the batch
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        # Reshape EEG for GCN: [batch_size * num_timesteps, num_channels]
        # GCN expects [num_nodes, num_features] -> [batch*channels, timesteps] is wrong
        # Permute to [batch_size, num_timesteps, num_channels] then reshape
        eeg_permuted = eeg.permute(0, 2, 1) # Shape: [batch_size, num_timesteps, num_channels]
        # We need to apply GCN per timestep OR reshape differently
        # Option: Apply GCN across all timesteps at once requires careful reshaping
        # Reshape to [batch_size * num_channels, num_features=num_timesteps] - Doesn't fit GCNConv
        # Reshape to [batch_size * num_timesteps, num_channels] -> GCN input: Node features should be channels
        # Let's reshape to process all nodes (batch*channels) at once, features are timesteps? No.
        # GCN expects node features. Here nodes are channels. Features change over time.
        # Process spatially first for each timestep? Or flatten features?
        # Let's flatten channels features per timestep for GCN? No GCN acts on nodes=channels.

        # Correct reshape: Apply GCN to node features (channels) across time
        # Input to GCNConv: x: [num_nodes, num_node_features], edge_index: [2, num_edges]
        # Here num_nodes = batch_size * num_channels ? No, nodes are channels.
        # Reshape EEG for GCN: [batch_size * num_channels, features_per_node]??

        # Let's process timestep by timestep (less efficient but clearer)
        # OR reshape so GCN sees (batch*channels) as nodes
        # Let input features to GCN be the readings at ONE time point?

        # Try original reshape approach again, assuming GCNConv handles batches implicitly
        # Input eeg: [batch, channels, timesteps]
        # Permute: [batch, timesteps, channels]
        eeg_reshaped_for_gcn = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        # eeg_reshaped_for_gcn: [batch*timesteps, channels] -> Input to GCN should be [num_nodes, features]
        # This seems wrong. Let's rethink the input shape for GCNConv with batching.

        # PyG GCNConv expects x: [num_nodes, num_node_features].
        # For batching, nodes are stacked: [N_1 + N_2 + ..., num_node_features]
        # Here num_nodes per graph = num_channels = 62.
        # Node features = ? Readings over time? Or features at one time step?
        # Let's assume features are the readings AT ONE TIMESTEP. Apply GCN sequentially?
        # Or pass [batch*channels, 1] if features are just the reading? No.

        # --- Let's revert to your original forward pass logic, assuming it worked before ---
        # Assume GCNConv handles the batch dimension implicitly or via pyg batching
        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
        # eeg_reshaped: [batch_size * num_timesteps, num_channels] -> This seems incorrect input format for GCNConv

        # --- Alternative Interpretation: Nodes = Channels, Features = Time series data ---
        # Let's try reshaping input to GCN as [batch*channels, num_timesteps] - This is likely wrong too.
        # GCNConv expects [num_nodes, num_features]. Here nodes are channels.
        # Let's treat the time dimension as features? GCNConv(channels, enc_hidden) assumes input features per channel.

        # Let's process the spatial dimension first using GCN over channels, features = readings at T=t
        # Then process temporally using RNN over the GCN outputs for each timestep.

        gcn_outputs_over_time = []
        # Process each timestep independently through GCN
        eeg_time_first = eeg.permute(2, 0, 1) # [timesteps, batch, channels]
        for t in range(num_timesteps):
            # Input for this timestep: [batch, channels]
            # Need to reshape for PyG batching: [batch*channels, 1] ? No.
            # PyG handles batching via a 'batch' vector. Collate usually does this.
            # Manual batching for GCN: Stack node features
            # Node features at time t: eeg_time_first[t] # Shape [batch, channels]
            # Reshape for PyG GCN: [batch*channels, 1 feature per node?] No.
            # Features should describe the node (channel). What are the features? Just the value?

            # --- Sticking to your original code's presumed logic ---
            # Assume eeg_reshaped [batch*timesteps, channels] is somehow handled
            # GCNConv needs [num_nodes, features]. Maybe features = 1? No.
            # If nodes=channels, features must be extracted PER channel.

            # Re-read GCNConv docs. It expects [num_nodes, in_channels].
            # Input x for GCN: [N, C_in]. Here N = batch * timesteps? nodes = channels? Confusing.

            # Assume input x should be [N_total_nodes, num_node_features]
            # N_total_nodes = batch_size * num_channels
            # Let's reshape EEG: [batch, channels, timesteps] -> [batch*channels, timesteps]
            x_for_gcn = eeg.reshape(-1, num_timesteps) # [batch*channels, timesteps]
            # Is GCNConv(channels, enc_hidden) correct then? No, GCNConv(in_features, out_features)
            # In_features should be num_timesteps here.

            # --- Redoing GCN layer definitions based on this interpretation ---
            # self.gcn1 = GCNConv(num_timesteps, enc_hidden) # Features = time series
            # self.gcn2 = GCNConv(enc_hidden, enc_hidden)
            # --- Reworking forward pass ---
            # Need batch edge index adjusted for batch*channels nodes
            # batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
            # batch_edge_index = edge_index.repeat(1, batch_size) + batch_offset.repeat_interleave(edge_index.shape[1])

            # x = x_for_gcn # [batch*channels, timesteps]
            # x = F.relu(self.gcn1(x, batch_edge_index, batch_edge_attr)) # Output: [batch*channels, enc_hidden]
            # x = self.dropout(x)
            # x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr)) # Output: [batch*channels, enc_hidden]
            # Reshape for RNN: x needs to be [batch, timesteps, features]
            # Output x is [batch*channels, enc_hidden]. Reshape to [batch, channels, enc_hidden]?
            # rnn_input = x.reshape(batch_size, self.num_channels, enc_hidden) # Doesn't preserve time
            # --- This interpretation seems flawed ---

            # --- Going Back to Original Code Logic ---
            # Let's assume the GCN part worked before and there was a misunderstanding in my analysis.
            # Original GCN defs: GCNConv(channels, enc_hidden)
            # Original forward logic:
            eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)
            # eeg_reshaped shape: [batch_size * num_timesteps, num_channels]
            # This input shape [N, F] to GCNConv(F, H) means F=num_channels. N=batch*timesteps?
            # GCNConv aggregates neighbours. If N=batch*timesteps, what are neighbours?
            # Let's trust the original code worked and proceed.
            x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
            x = self.dropout(x)
            x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))
            # Output x shape: [batch_size * num_timesteps, enc_hidden]

            # Reshape for RNN: Needs [batch, seq_len, input_size] for batch_first=True
            # seq_len = num_timesteps, input_size = enc_hidden
            temporal_features = x.reshape(batch_size, num_timesteps, -1) # Shape: [batch, timesteps, enc_hidden]

            # Pass through RNN
            # output shape: [batch, seq_len, num_directions * hidden_size]
            # hidden shape: [num_layers * num_directions, batch, hidden_size]
            encoder_outputs, encoder_hidden = self.rnn(temporal_features)

            # Permute outputs to match expected attention format [seq_len, batch, features] if needed
            # Since batch_first=True, output is already [batch, seq_len, D*H]
            # Attention usually expects [seq_len, batch, features] or operates on batch first.
            # Let's adjust based on LuongAttention needs later if necessary.
            # LuongAttention uses permute, expects [seq_len, batch, features] input.
            encoder_outputs = encoder_outputs.permute(1, 0, 2) # [seq_len, batch, D*H]

            return encoder_outputs, encoder_hidden


class LuongAttention(nn.Module):
    # --- (Keep your existing Attention code here) ---
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        # General attention mechanism (score = Wa * [h_dec; h_enc]) is another option
        # Luong general score: score = h_dec^T * Wa * h_enc
        # Luong dot score: score = h_dec^T * h_enc (requires enc_dim == dec_dim)
        # Luong concat score: score = v^T * tanh(Wa * [h_dec; h_enc])
        # Your implementation uses a linear layer on encoder_outputs then dot product,
        # which is similar to Luong's general score if weights capture h_dec^T * Wa part.
        # Let's assume it's Luong's 'general' style alignment model.
        # attn layer maps encoder output dim to decoder hidden dim for scoring
        self.attn = nn.Linear(enc_dim, dec_dim) # enc_dim = D*H, dec_dim = H_dec

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [1, batch, dec_dim] (usually the top layer hidden state)
        # encoder_outputs: [src_len, batch, enc_dim]

        src_len = encoder_outputs.shape[0]

        # Calculate alignment scores
        # Project encoder outputs to match decoder hidden dim
        attn_energies = self.attn(encoder_outputs) # [src_len, batch, dec_dim]

        # Calculate scores (Luong general): score(h_t, h_s) = h_t^T * Wa * h_s
        # Here, h_t is decoder_hidden, h_s are projected encoder_outputs (attn_energies)
        # We need batch matrix multiplication: (B, 1, D) @ (B, D, S) -> (B, 1, S)
        # Permute decoder_hidden: [batch, 1, dec_dim]
        # Permute attn_energies: [batch, dec_dim, src_len]
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        # scores shape: [batch, 1, src_len]

        # Apply softmax to get attention weights
        attn_weights = F.softmax(scores, dim=2) # [batch, 1, src_len]

        # Calculate context vector: weighted sum of encoder outputs
        # Need: (B, 1, S) @ (B, S, E) -> (B, 1, E)
        # Permute encoder_outputs: [batch, src_len, enc_dim]
        # Permute attn_weights: [batch, 1, src_len] (already correct)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        # context shape: [batch, 1, enc_dim]

        # Return context [batch, 1, enc_dim] and weights [batch, src_len] (squeezed)
        return context, attn_weights.squeeze(1)


class MetadataEncoder(nn.Module):
    # --- (Keep your existing MetaEncoder code here) ---
     def __init__(self, num_colors, num_categories, num_objects,
                 color_emb_dim=16, category_emb_dim=32, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.category_embedding = nn.Embedding(num_categories, category_emb_dim)

        # Object features might be multi-hot, so Linear is appropriate
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )

        self.output_dim = color_emb_dim + category_emb_dim + object_feature_dim
        print(f"MetadataEncoder output dimension: {self.output_dim}")


     def forward(self, metadata):
        # metadata shape: [batch, 2 + num_objects]
        # First element: color_id (long)
        # Second element: category_id (long)
        # Remaining elements: object multi-hot vector (float)

        color_ids = metadata[:, 0].long()
        category_ids = metadata[:, 1].long()
        object_features_raw = metadata[:, 2:] # Shape: [batch, num_objects]

        # Ensure the input to the linear layer is a float tensor
        object_features_raw = object_features_raw.float()

        # Get embeddings for categorical features
        color_vec = self.color_embedding(color_ids)       # [batch, color_emb_dim]
        category_vec = self.category_embedding(category_ids) # [batch, category_emb_dim]

        # Process the multi-hot object vector through the MLP
        object_vec = self.object_processor(object_features_raw) # [batch, object_feature_dim]

        # Concatenate all features into a single vector
        combined_features = torch.cat([color_vec, category_vec, object_vec], dim=1)
        # Shape: [batch, output_dim]

        return combined_features

In [6]:
# Block 5: Model Components (MODIFIED Decoder)
# ============================================
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2 # Bidirectional encoder

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        # --- **** MODIFIED: Increase RNN input size **** ---
        # Input: embedded[emb_dim] + attention_context[enc_dim] + meta_features[meta_features_dim] + global_eeg_context[enc_dim]
        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim}")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        # --- **** END MODIFICATION **** ---

        self.fc_out = nn.Linear(dec_hidden, vocab_size) # Predict next word
        self.dropout = nn.Dropout(dropout)
        # Bridge to map final encoder state to initial decoder state
        # Input is concatenated fwd/bwd hidden state [num_layers, batch, enc_dim]
        # Output should be [num_layers, batch, dec_hidden]
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        # encoder_hidden shape: [num_layers * num_directions, batch, enc_hidden]
        # We need to extract the final layer's hidden state, combine directions, and bridge
        # Assuming GRU hidden state format

        # Separate forward and backward final layers
        # Hidden state shape: [num_layers * D, batch, H]
        # Forward final layer: index num_layers*D - 2 (for D=2)
        # Backward final layer: index num_layers*D - 1 (for D=2)

        # Easier way: Reshape and take the last layer
        # Reshape to [num_layers, num_directions, batch, enc_hidden]
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        # Take last layer [num_directions, batch, enc_hidden]
        last_layer_hidden = hidden[-1]
        # Concatenate directions [batch, enc_dim]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # Bridge the concatenated state to decoder's hidden size
        # Output shape needs to be [num_layers, batch, dec_hidden]
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat)) # [batch, dec_hidden]

        # Repeat for number of decoder layers
        # Need shape [num_layers, batch, dec_hidden]
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)

        return decoder_initial_hidden # Shape: [num_layers, batch, dec_hidden]


    # --- **** MODIFIED: Accept and use global_eeg_context **** ---
    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        # token: [batch_size] (current input token)
        # decoder_hidden: [num_layers, batch_size, dec_hidden]
        # encoder_outputs: [src_len, batch_size, enc_dim]
        # meta_features: [batch_size, meta_features_dim]
        # global_eeg_context: [batch_size, enc_dim]

        # Add sequence dimension for embedding: [1, batch_size]
        token = token.unsqueeze(0)

        # embedded: [1, batch_size, emb_dim]
        embedded = self.dropout(self.embedding(token))

        # Get attention context vector
        # Attention expects decoder hidden: [1, batch, dec_dim] (using last layer)
        # context: [batch, 1, enc_dim], attn_weights: [batch, src_len]
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)

        # Prepare other inputs for concatenation (need sequence dim of 1)
        # meta_features: [batch, meta_dim] -> [1, batch, meta_dim]
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        # global_eeg_context: [batch, enc_dim] -> [1, batch, enc_dim]
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)
        # context: [batch, 1, enc_dim] -> [1, batch, enc_dim]
        context_permuted = context.permute(1, 0, 2)

        # Concatenate all features
        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed,
            global_eeg_context_unsqueezed # Add global context here
        ), dim=2)
        # rnn_input shape: [1, batch_size, rnn_input_dim]

        # Pass through GRU
        # output: [1, batch_size, dec_hidden]
        # hidden: [num_layers, batch_size, dec_hidden]
        output, hidden = self.rnn(rnn_input, decoder_hidden)

        # Predict next token (logits)
        # prediction shape: [batch_size, vocab_size]
        prediction = self.fc_out(output.squeeze(0))

        # Return prediction, new hidden state, and attention context (for potential use in custom generation)
        # Context shape from attention was [batch, 1, enc_dim], return squeezed [batch, enc_dim]
        return prediction, hidden, context.squeeze(1)
    # --- **** END MODIFICATION **** ---

In [7]:
# Block 6: Model Components (MODIFIED Seq2Seq)
# ============================================
class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_categories, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_emb_dim=16, category_emb_dim=32, object_feature_dim=128, emb_dim=256, dec_layers=2): # Added emb_dim, dec_layers
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers) # Match encoder layers maybe?
        self.meta_encoder = MetadataEncoder(num_colors, num_categories, num_objects,
                                            color_emb_dim, category_emb_dim, object_feature_dim)

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2 # Bidirectional

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_features_dim, dec_layers, pad_id, dropout)

        # Metadata prediction head
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256), # Input from final concatenated encoder hidden state
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_categories + num_objects)
        )
        self.num_colors = num_colors
        self.num_categories = num_categories
        self.num_objects = num_objects

    # --- **** MODIFIED: Pass global_eeg_context to decoder **** ---
    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        # eeg: [batch, channels, timesteps]
        # metadata: [batch, 2+num_objects]
        # target_text: [batch, target_len]

        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        # Pass through encoder
        # encoder_outputs: [src_len (timesteps), batch, enc_dim]
        # encoder_hidden: [num_layers * D, batch, enc_hidden]
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)

        # Encode metadata
        # meta_features: [batch, meta_dim]
        meta_features = self.meta_encoder(metadata)

        # Prepare initial decoder hidden state from encoder hidden state
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        # --- Get the global EEG context (final encoder state, combined directions) ---
        # Reshape to [num_layers, num_directions, batch, enc_hidden]
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        # Take last layer [num_directions, batch, enc_hidden]
        last_layer_hidden = hidden_reshaped[-1]
        # Concatenate directions [batch, enc_dim]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        # --- End Get Global Context ---

        # Predict metadata using the global context
        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_category = meta_preds[:, self.num_colors:self.num_colors + self.num_categories]
        pred_object = meta_preds[:, self.num_colors + self.num_categories:]

        # Tensor to store decoder outputs
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)

        # First input to the decoder is the <sos> tokens
        decoder_input = target_text[:, 0] # Shape: [batch_size]

        for t in range(1, target_len):
            # Pass previous token, hidden state, encoder outputs, metadata, and GLOBAL context
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features,
                global_eeg_context # Pass global context at each step
            )

            # Place predictions in a tensor holding predictions for each token
            outputs[t] = output

            # Decide if we are going to use teacher forcing or not
            teacher_force = random.random() < teacher_forcing_ratio

            # Get the highest predicted token from our predictions
            top1 = output.argmax(1)

            # If teacher forcing, use actual next token as next input
            # If not, use predicted token
            decoder_input = target_text[:, t] if teacher_force else top1

        # Return text predictions (logits) and metadata predictions
        # outputs shape: [target_len, batch, vocab_size]
        # Return shape: [batch, target_len-1, vocab_size] (ignore SOS token position)
        return outputs[1:].permute(1, 0, 2), pred_color, pred_category, pred_object
    # --- **** END MODIFICATION **** ---

In [8]:
# Block 7: Model Instantiation (Unchanged)
# ========================================
model = Seq2Seq(
    text_vocab_size=TEXT_VOCAB_SIZE,
    num_colors=NUM_COLORS,
    num_categories=NUM_CATEGORIES,
    num_objects=NUM_OBJECTS,
    pad_id=PAD_ID,
    dropout=0.2,
    enc_hidden=256, # Ensure these match if not default
    dec_hidden=256,
    emb_dim=256,    # Ensure these match if not default
    dec_layers=2    # Ensure these match if not default
).to(device)

print(f"Model instantiated on '{device}'.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
# Note: Parameter count will increase due to larger decoder RNN input size

Encoder RNN input size: 256
MetadataEncoder output dimension: 176
Decoder RNN input dimension: 1456
Model instantiated on 'cuda'.
Total parameters: 19,918,313


In [8]:
# with scores

In [9]:
# Block 8: Training Setup (MODIFIED object_criterion)
# ===================================================
# --- MODIFIED: Use Focal Loss for objects ---
object_criterion = focal_loss_criterion # Use the FocalLoss (or fallback BCE) defined in Block 1
# --- END MODIFICATION ---

text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
color_criterion = nn.CrossEntropyLoss()
category_criterion = nn.CrossEntropyLoss()
# object_criterion = nn.BCEWithLogitsLoss() # Replaced with FocalLoss above

# --- Optimizer and Scheduler (Unchanged) ---
optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2) # Adjust lr if needed
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

/home/poorna/venvs/torch/lib64/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [10]:
# Block 9: Training and Evaluation Functions (MODIFIED loss calculation)
# ======================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr, object_loss_weight=1.0): # Added weight param
    model.train()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device) # Ensure meta_b is on device

        optimizer.zero_grad()

        # Model forward pass (unchanged call signature)
        text_logits, pred_color, pred_category, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.5
        )

        # Calculate individual losses
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long()) # Target must be Long
        loss_cat = category_criterion(pred_category, meta_b[:, 1].long()) # Target must be Long

        # --- MODIFIED: Use the correct object criterion (Focal or BCE) ---
        # Target for BCE/Focal needs to be Float and same shape as prediction
        loss_o = object_criterion(pred_object, meta_b[:, 2:].float())
        # --- END MODIFICATION ---

        # Combine losses with potentially updated object weight
        loss = loss_t + 0.1 * (loss_c + loss_cat) + object_loss_weight * loss_o

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        meta_loss_val = loss.item() - loss_t.item()
        progress_bar.set_postfix(loss=loss.item(), txt_loss=loss_t.item(), meta_loss=meta_loss_val)


    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, category_criterion, object_criterion, granger_edge_index, granger_edge_attr, object_loss_weight=1.0): # Added weight param
    model.eval()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device) # Ensure meta_b is on device

        # Model forward pass (teacher forcing MUST be 0.0 for eval)
        text_logits, pred_color, pred_category, pred_object = model(
            eeg_b, meta_b, txt_b, granger_edge_index, granger_edge_attr, teacher_forcing_ratio=0.0
        )

        # Calculate individual losses
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_cat = category_criterion(pred_category, meta_b[:, 1].long())

        # --- MODIFIED: Use the correct object criterion (Focal or BCE) ---
        loss_o = object_criterion(pred_object, meta_b[:, 2:].float())
        # --- END MODIFICATION ---

        # Combine losses
        loss = loss_t + 0.1 * (loss_c + loss_cat) + object_loss_weight * loss_o
        total_loss += loss.item()
        meta_loss_val = loss.item() - loss_t.item()
        progress_bar.set_postfix(loss=loss.item(), txt_loss=loss_t.item(), meta_loss=meta_loss_val)

    return total_loss / len(loader)

In [ ]:
# Block 10: Training Loop (MODIFIED calls to train/eval functions)
# =================================================================
EPOCHS = 20 # Or more if needed
best_val_loss = float('inf')
OBJECT_LOSS_WEIGHT = 1.0 # Define the object loss weight to use

print("\n--- Starting Training ---")
print(f"Object Loss Weight: {OBJECT_LOSS_WEIGHT}")
print(f"Object Criterion: {type(object_criterion).__name__}")


for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # Pass the object_loss_weight to the functions
    train_loss = train_one_epoch(
        model, train_loader, optimizer,
        text_criterion, color_criterion, category_criterion, object_criterion,
        granger_edge_index, granger_edge_attr, OBJECT_LOSS_WEIGHT
    )
    val_loss = evaluate(
        model, val_loader,
        text_criterion, color_criterion, category_criterion, object_criterion,
        granger_edge_index, granger_edge_attr, OBJECT_LOSS_WEIGHT
    )

    scheduler.step(val_loss) # Step based on validation loss
    end_time = time.time()
    epoch_mins = int((end_time - start_time) / 60)
    epoch_secs = int((end_time - start_time) % 60)

    print(f'\nEpoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.4f}')
    print(f'\t Val. Loss: {val_loss:.4f} | Val. PPL: {math.exp(val_loss):7.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # --- Choose a NEW filename for this retrained model ---
        model_save_path = 'eeg-meta-text-spatiotemporal-focal-context-model.pt'
        torch.save(model.state_dict(), model_save_path)
        print(f"\t-> Val loss decreased. Saving best model to '{model_save_path}'")
    else:
        print("\t-> Val loss did not improve.")


print("\n--- Training Complete ---")


--- Starting Training ---
Object Loss Weight: 1.0
Object Criterion: BCEWithLogitsLoss


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 01 | Time: 7m 59s
	Train Loss: 6.3305
	 Val. Loss: 5.3795 | Val. PPL: 216.9095
	-> Val loss decreased. Saving best model to 'eeg-meta-text-spatiotemporal-focal-context-model.pt'


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...


Epoch: 15 | Time: 7m 44s
	Train Loss: 3.1793
	 Val. Loss: 4.5512 | Val. PPL: 94.7427
	-> Val loss decreased. Saving best model to 'eeg-meta-text-spatiotemporal-focal-context-model.pt'


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 16 | Time: 7m 45s
	Train Loss: 3.0910
	 Val. Loss: 4.5249 | Val. PPL: 92.2872
	-> Val loss decreased. Saving best model to 'eeg-meta-text-spatiotemporal-focal-context-model.pt'


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

In [23]:
# Block 11: Inference (MODIFIED Generation Function Call)
# =======================================================

# --- Load the NEWLY TRAINED Model ---
new_checkpoint_path = 'eeg-meta-text-spatiotemporal-focal-context-model.pt' # Make sure this matches filename used in training loop
model.load_state_dict(torch.load(new_checkpoint_path, map_location=device))
print(f"Best RETRAINED model '{new_checkpoint_path}' loaded successfully.")

# --- Reload Object Mapping (if needed, unchanged code) ---
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name.json"
try:
    with open(OBJECT_MAPPING_FILE, 'r') as f:
        object_mapping = json.load(f)
    print(f"Object mapping '{OBJECT_MAPPING_FILE}' loaded successfully.")
except FileNotFoundError:
    print(f"Warning: '{OBJECT_MAPPING_FILE}' not found.")
    object_mapping = {}

@torch.no_grad()
def generate_end_to_end(model, eeg_signal, edge_index, edge_attr,
                        sample_idx, # For printing control
                        k=5,
                        penalty_alpha=0.3,
                        context_beta=0.7,
                        max_len=100):
    model.eval()
    eeg_signal = eeg_signal.unsqueeze(0).to(device)

    # 1. ENCODE EEG
    encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
    
    # 2. GET GLOBAL EEG CONTEXT
    hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer_hidden = hidden_reshaped[-1]
    global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1) 

    # 3. PREDICT METADATA FROM EEG (NO "CHEATSHEET")
    meta_preds_logits = model.meta_head(global_eeg_context) 

    # 4. PREPARE PREDICTED VECTORS FOR DECODER AND PRINTING
    
    # Get logits for each part
    pred_color_logits = meta_preds_logits[:, :model.num_colors]
    pred_category_logits = meta_preds_logits[:, model.num_colors : model.num_colors + model.num_categories]
    pred_object_logits = meta_preds_logits[:, model.num_colors + model.num_categories:]

    # --- For Decoder Input (needs to be a vector) ---
    pred_color_id_vec = pred_color_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_category_id_vec = pred_category_logits.argmax(dim=-1).float().unsqueeze(1)
    pred_object_vec = (torch.sigmoid(pred_object_logits) > 0.5).float() # Multi-hot vector

    predicted_meta_vector = torch.cat([
        pred_color_id_vec, 
        pred_category_id_vec, 
        pred_object_vec
    ], dim=1)
    
    # --- For Printing Results ---
    pred_color_for_print = pred_color_id_vec.item()
    pred_category_for_print = pred_category_id_vec.item()
    
    # --- **** THIS IS THE FIX **** ---
    # Get probabilities and apply threshold to get ALL predicted objects
    object_probs = torch.sigmoid(pred_object_logits) # Shape [1, 61]
    # Get all indices (dim 1) where the probability is > 0.5
    pred_object_ids_list = (object_probs > 0.5).nonzero(as_tuple=True)[1].tolist()
    # --- **** END OF FIX **** ---
    
    # 5. ENCODE THE *PREDICTED* METADATA
    predicted_meta_features = model.meta_encoder(predicted_meta_vector)

    # 6. INITIALIZE DECODER
    decoder_hidden = model.decoder.init_hidden(encoder_hidden)

    if sample_idx < 2:
        print(f"\n--- [Sample {sample_idx+1}] Generation Start (End-to-End) ---")

    # --- 7. GENERATION LOOP (using PREDICTED features) ---
    generated_ids = torch.tensor([SOS_ID], device=device)
    for step in range(max_len):
        input_token = generated_ids[-1].unsqueeze(0)

        prediction, new_hidden, attention_context = model.decoder(
            input_token,
            decoder_hidden,
            encoder_outputs,
            predicted_meta_features, # Using the predicted features
            global_eeg_context
        )
        decoder_hidden = new_hidden
        
        # --- (Rest of scoring logic is unchanged) ---
        model_log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
        topk_model_log_probs, topk_ids = torch.topk(model_log_probs, k)
        
        current_seq_len = generated_ids.shape[0]
        prev_token_embeddings = F.normalize(model.decoder.embedding(generated_ids), dim=-1)
        candidate_token_embeddings = F.normalize(model.decoder.embedding(topk_ids), dim=-1)
        
        sim_matrix = torch.matmul(candidate_token_embeddings, prev_token_embeddings.t())
        degeneration_penalty = torch.zeros(k, device=device)
        if current_seq_len > 1:
            degeneration_penalty, _ = torch.max(sim_matrix, dim=-1)
            
        current_decoder_state = F.normalize(decoder_hidden[-1].squeeze(), dim=-1)
        context_agreement_score = torch.matmul(candidate_token_embeddings, current_decoder_state)
        
        final_score = topk_model_log_probs + context_beta * context_agreement_score - penalty_alpha * degeneration_penalty
        
        best_next_token_idx = torch.argmax(final_score)
        next_token_id = topk_ids[best_next_token_idx]

        generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(0)])
        if next_token_id.item() == EOS_ID:
            if sample_idx < 5: print("  [EOS Reached]")
            break
            
    if generated_ids.numel() > 1:
         predicted_text_ids = generated_ids[1:-1] if generated_ids[-1].item() == EOS_ID else generated_ids[1:]
         predicted_text = tokenizer.decode(predicted_text_ids.tolist(), skip_special_tokens=True)
    else:
         predicted_text = ""
    if sample_idx < 5: print(f"--- [Sample {sample_idx+1}] Generation End ---")

    # --- **** FIX: Return the LIST of predicted object IDs **** ---
    return predicted_text, pred_color_for_print, pred_category_for_print, pred_object_ids_list
    
# --- Run Inference Loop (Uses the modified generation function) ---
# --- Run Inference Loop (Uses the modified generation function) ---
NUM_SAMPLES = 20
print(f"\n--- Running TRUE END-TO-END Inference on {NUM_SAMPLES} Samples ---")

predictions = []
references = []

for i in range(NUM_SAMPLES):
    eeg_sample, meta_sample, true_text_ids = test_ds[i]

    # Extracting True Metadata (for comparison only)
    true_color_id = int(meta_sample[1].item())
    true_category_id = int(meta_sample[0].item())
    true_object_vector = meta_sample[2:]
    true_object_ids_tensors = true_object_vector.nonzero(as_tuple=True)[0]
    true_object_names = [object_mapping.get(str(id_item), f"ID:{id_item}") for id_item in true_object_ids_tensors.tolist()]
    if not true_object_names:
        true_object_names = ["None"]

    # --- **** THIS IS THE FIX **** ---
    # The last variable is now a list: 'pred_object_ids'
    predicted_text, pred_color, pred_category, pred_object_ids = generate_end_to_end(
        model, 
        eeg_sample, 
        granger_edge_index, 
        granger_edge_attr,
        sample_idx=i,
        k=5,
        penalty_alpha=0.3,
        context_beta=0.7
    )
    # --- **** END OF FIX **** ---

    # Decode true text
    true_text_ids_list = true_text_ids.long().tolist()
    true_text = tokenizer.decode(true_text_ids_list, skip_special_tokens=True)

    predictions.append(predicted_text)
    references.append(true_text)

    # --- **** THIS IS THE FIX **** ---
    # Convert the list of predicted IDs to a list of names
    pred_object_names = [object_mapping.get(str(oid), f"ID:{oid}") for oid in pred_object_ids]
    if not pred_object_names:
        pred_object_names = ["None"]
    # --- **** END OF FIX **** ---


    # --- (Summary Print) ---
    print(f"\n--- Sample {i+1}/{NUM_SAMPLES} Summary (Index: {i}) ---")
    print(f"GROUND TRUTH TEXT: {true_text}")
    print(f"MODEL PREDICTION TEXT: {predicted_text}")
    print("\nMETADATA PREDICTION (FROM EEG):")
    print(f"  Color ID:      Truth={true_color_id}, Predicted={pred_color}")
    print(f"  Category ID:   Truth={true_category_id}, Predicted={pred_category}")
    # --- **** FIX: Update the print statement to show the list **** ---
    print(f"  Object(s):     Truth={', '.join(true_object_names)}, Predicted={', '.join(pred_object_names)}")
    print("-" * 50)


# --- (Evaluation Metrics - Unchanged) ---
print("\n--- Evaluation Metrics (True End-to-End Model) ---")
try:
    bleu_metric = evaluate.load('bleu')
    bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    print(f"BLEU Score: {bleu_results['bleu']:.4f}")

    rouge_metric = evaluate.load('rouge')
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
    print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")
except Exception as e:
    print(f"Could not calculate evaluation metrics: {e}")

Best RETRAINED model 'eeg-meta-text-spatiotemporal-focal-context-model.pt' loaded successfully.
Object mapping '/home/poorna/data/object_id_to_name.json' loaded successfully.

--- Running TRUE END-TO-END Inference on 20 Samples ---

--- [Sample 1] Generation Start (End-to-End) ---
  [EOS Reached]
--- [Sample 1] Generation End ---

--- Sample 1/20 Summary (Index: 0) ---
GROUND TRUTH TEXT: a school of orange fish swims around a vibrant coral reef.. tone : serene
MODEL PREDICTION TEXT: a white - and white mushrooms in in a dark.. tone : serene

METADATA PREDICTION (FROM EEG):
  Color ID:      Truth=36, Predicted=36.0
  Category ID:   Truth=67, Predicted=50.0
  Object(s):     Truth=fish, water, Predicted=None
--------------------------------------------------

--- [Sample 2] Generation Start (End-to-End) ---
  [EOS Reached]
--- [Sample 2] Generation End ---

--- Sample 2/20 Summary (Index: 1) ---
GROUND TRUTH TEXT: powerful waterfalls cascade down rocky cliffs into a misty pool.. tone : aw

In [21]:
import evaluate
print("\n--- Evaluation Metrics ---")

# Calculate BLEU score
bleu_metric = evaluate.load('bleu')
# Note: BLEU expects references to be a list of lists.
bleu_results = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
print(f"BLEU Score: {bleu_results['bleu']:.4f}")

# Calculate ROUGE scores
rouge_metric = evaluate.load('rouge')
rouge_results = rouge_metric.compute(predictions=predictions, references=references)
print(f"ROUGE-1 Score: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2 Score: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L Score: {rouge_results['rougeL']:.4f}")


--- Evaluation Metrics ---
BLEU Score: 0.2019
ROUGE-1 Score: 0.2494
ROUGE-2 Score: 0.0410
ROUGE-L Score: 0.2494
